In [1]:
import os
import numpy as np
import pickle
import librosa
import scipy.signal
from scipy.stats import kurtosis
from xgboost import XGBClassifier

# ==========================================
# 1. FEATURE EXTRACTION FUNCTIONS
# ==========================================
def preprocess_audio(audio_path, target_sr=16000):
    y, sr = librosa.load(audio_path, sr=target_sr, mono=True)
    y_trimmed, _ = librosa.effects.trim(y, top_db=20)
    return y_trimmed, sr

def extract_glottal_residual_sharpness(y):
    y_pre = librosa.effects.preemphasis(y)
    lpc_coeffs = librosa.lpc(y_pre, order=16)
    residual = scipy.signal.lfilter(lpc_coeffs, [1.0], y_pre)
    return float(kurtosis(residual)), float(np.std(residual))

def extract_spectral_flux_dynamics(y):
    S = np.abs(librosa.stft(y))
    flux = np.maximum(0, np.diff(S, axis=1))
    total_flux_per_frame = np.sum(flux, axis=0)
    return float(np.std(total_flux_per_frame)), float(np.max(total_flux_per_frame)) 

def extract_modulation_energy(y, sr):
    analytic_signal = scipy.signal.hilbert(y)
    amplitude_envelope = np.abs(analytic_signal)
    env_fft = np.abs(np.fft.rfft(amplitude_envelope))
    freqs = np.fft.rfftfreq(len(amplitude_envelope), 1/sr)
    muscle_band = np.where((freqs >= 2) & (freqs <= 10))[0]
    if len(muscle_band) == 0:
        return 0.0
    return float(np.mean(env_fft[muscle_band]))

def extract_vocal_tract_formants(y, sr):
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    delta_mfccs = librosa.feature.delta(mfccs)
    delta2_mfccs = librosa.feature.delta(mfccs, order=2)
    combined_mfccs = np.vstack([mfccs, delta_mfccs, delta2_mfccs])
    return np.hstack([np.mean(combined_mfccs, axis=1), np.std(combined_mfccs, axis=1)]) 

def extract_aliasing_signature(y, sr):
    stft_matrix = np.abs(librosa.stft(y))
    freqs = librosa.fft_frequencies(sr=sr)
    fold_back_indices = np.where((freqs >= 5000) & (freqs <= 8000))[0]
    band_rms_per_frame = np.sqrt(np.mean(stft_matrix[fold_back_indices, :]**2, axis=0))
    return float(np.mean(band_rms_per_frame)), float(np.std(band_rms_per_frame))

def extract_spectral_rolloff_variance(y, sr):
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.95)[0]
    return float(np.std(rolloff))

def extract_zcr_variance(y):
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    return float(np.std(zcr))

def build_advanced_feature_vector(audio_path):
    """Compiles the targeted feature array with MFCCs FIRST, then physics metrics."""
    y, sr = preprocess_audio(audio_path, target_sr=16000)
    
    glottal_kurtosis, glottal_std = extract_glottal_residual_sharpness(y)
    
    # Ignore the second return value (flux_max) by assigning it to '_'
    flux_std, _ = extract_spectral_flux_dynamics(y) 
    
    mod_energy = extract_modulation_energy(y, sr)
    rolloff_std = extract_spectral_rolloff_variance(y, sr)
    zcr_std = extract_zcr_variance(y)
    alias_mean, alias_std = extract_aliasing_signature(y, sr)
    
    formant_features = extract_vocal_tract_formants(y, sr)
    
    # Assemble Vector (MFCCs first, then Physics)
    feature_vector = np.hstack((
        formant_features,                          # Indices 0 to 119 (120 MFCC features)
        [glottal_kurtosis, glottal_std],           # Indices 120, 121
        [flux_std],                                # Index 122
        [mod_energy],                              # Index 123
        [rolloff_std],                             # Index 124
        [zcr_std],                                 # Index 125
        [alias_mean, alias_std]                    # Indices 126, 127
    ))
    return feature_vector


# ==========================================
# 2. THE INFERENCE ENGINE
# ==========================================
def analyze_new_audio_xgb(audio_path, model_mfcc, model_physics, scaler_physics, fake_threshold=0.50):
    print(f"\n--- ANALYZING: {os.path.basename(audio_path)} ---")
    
    try:
        # Extract features
        full_vector = build_advanced_feature_vector(audio_path)
    except Exception as e:
        return f"ERROR: Could not process audio. {e}"

    # Slice the array into the two towers (NEW CLEAN SLICING LOGIC)
    vec_mfcc = full_vector[0:120].reshape(1, -1)
    vec_physics = full_vector[120:].reshape(1, -1)

    # --- DOOR ONE: Biometric Gate ---
    # I noticed you commented this out in your code. I'll leave it commented for you,
    # but the vec_mfcc variable is sliced correctly above if you ever want to re-enable it!
    
    #biometric_match = model_mfcc.predict(vec_mfcc)[0]
    #if biometric_match == 1: 
    #    return "❌ ACCESS DENIED: Voice identity does not match the target."
    #print("✅ IDENTITY VERIFIED: Passing to Anti-Spoofing Physics Gate...")

    # --- DOOR TWO: Physics Gate ---
    # We MUST scale the physics features using the loaded scaler!
    vec_physics_scaled = scaler_physics.transform(vec_physics)
    
    # Get the probability of it being a deepfake (Class 1)
    fake_probability = model_physics.predict_proba(vec_physics_scaled)[0][1]
    
    print(f"Detected Fake Probability: {fake_probability:.4f} (Threshold: {fake_threshold})")
    
    if fake_probability >= fake_threshold:
        return "🚨 THREAT DETECTED: Identity matched, but the audio is synthetically generated (Deepfake)."
    else:
        return "✅ ACCESS GRANTED: Genuine human voice verified."
    


# ==========================================
# 3. EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    print("Loading saved XGBoost models and scaler...")
    
    # Ensure these paths point to wherever you saved the .pkl files
    # from your training notebook!
    try:
        with open('tower1_mfcc_xgb.pkl', 'rb') as f:
            loaded_mfcc_xgb = pickle.load(f)
            
        with open('tower2_physics_xgb.pkl', 'rb') as f:
            loaded_phys_xgb = pickle.load(f)
            
        with open('physics_scaler.pkl', 'rb') as f:
            loaded_scaler = pickle.load(f)
            
        print("✅ Models and scaler loaded successfully!")
        
        # --- TEST YOUR AUDIO FILE HERE ---
        # Update this path to the audio file you want to test
        test_file = "/Users/vedsheth/Downloads/Spoof-Detection/try1.wav" 
        
        if os.path.exists(test_file):
            result = analyze_new_audio_xgb(
                audio_path=test_file, 
                model_mfcc=loaded_mfcc_xgb, 
                model_physics=loaded_phys_xgb, 
                scaler_physics=loaded_scaler,
                fake_threshold=0.50
            )
            print("\nRESULT:", result)
        else:
            print(f"File not found: {test_file}. Please check the path.")
            
    except FileNotFoundError as e:
        print(f"\n❌ Error loading files: {e}")
        print("Make sure 'tower1_mfcc_xgb.pkl', 'tower2_physics_xgb.pkl', and 'physics_scaler.pkl' are in the same folder as this notebook.")

Loading saved XGBoost models and scaler...
✅ Models and scaler loaded successfully!

--- ANALYZING: try1.wav ---
Detected Fake Probability: 0.0013 (Threshold: 0.5)

RESULT: ✅ ACCESS GRANTED: Genuine human voice verified.


In [2]:
import os

test_files = {
    "r1.wav": "REAL", "r2.wav": "REAL", "r3.wav": "REAL", "r4.wav": "REAL",
    "r5.wav": "REAL", "r6.wav": "REAL", "r7.wav": "REAL", "r8.wav": "REAL",
    "try1.wav": "SPOOF", "try2.wav": "SPOOF", "try3.wav": "SPOOF", "try4.wav": "SPOOF",
    "try5.wav": "SPOOF", "try6.wav": "SPOOF", "try7.wav": "SPOOF", "try8.wav": "SPOOF",
}

BASE = "/Users/vedsheth/Downloads/Spoof-Detection/"

print(f"{'File':<15} {'True Label':<10} {'Spoof%':>8} {'Verdict':<10} {'Correct?'}")
print("-" * 55)

correct = 0
for filename, true_label in test_files.items():
    path = os.path.join(BASE, filename)
    if not os.path.exists(path):
        print(f"{filename:<15} FILE NOT FOUND")
        continue
    fv = build_advanced_feature_vector(path)
    vec_physics = fv[120:].reshape(1, -1)
    vec_physics_scaled = loaded_scaler.transform(vec_physics)
    prob_spoof = loaded_phys_xgb.predict_proba(vec_physics_scaled)[0][1]
    verdict = "SPOOF" if prob_spoof >= 0.5 else "REAL"
    is_correct = verdict == true_label
    if is_correct:
        correct += 1
    print(f"{filename:<15} {true_label:<10} {prob_spoof*100:>7.1f}% {verdict:<10} {'✓' if is_correct else '✗'}")

print("-" * 55)
print(f"Overall: {correct}/16 = {correct/16*100:.1f}%")

File            True Label   Spoof% Verdict    Correct?
-------------------------------------------------------
r1.wav          REAL           0.1% REAL       ✓
r2.wav          REAL           0.0% REAL       ✓
r3.wav          REAL           0.0% REAL       ✓
r4.wav          REAL           0.0% REAL       ✓
r5.wav          REAL           0.0% REAL       ✓
r6.wav          REAL           0.2% REAL       ✓
r7.wav          REAL           0.3% REAL       ✓
r8.wav          REAL           0.0% REAL       ✓
try1.wav        SPOOF          0.1% REAL       ✗
try2.wav        SPOOF         93.1% SPOOF      ✓
try3.wav        SPOOF          0.1% REAL       ✗
try4.wav        SPOOF          0.0% REAL       ✗
try5.wav        SPOOF          0.1% REAL       ✗
try6.wav        SPOOF          0.0% REAL       ✗
try7.wav        SPOOF          0.0% REAL       ✗
try8.wav        SPOOF          0.1% REAL       ✗
-------------------------------------------------------
Overall: 9/16 = 56.2%
